# #69 2016~2024 시도별 총세출 분모 패널 구축

## tl;dr

- 주 기준은 지역통합 3개 회계 당초예산 순계이며, 17개 지역×9개년 153행이다.
- 일반회계·총계·광역본청·기금 포함 등 5개 분모 정의 765행을 보존한다.
- 대안별 키 중복·결측·0 이하 금액은 모두 0건이다.

## Context & Methods

지방재정365 기능별·회계별 자료와 세출예산 자료를 연도×지역으로 집계한다.

### Key Assumptions

- 주 기준은 일반·기타특별·공기업특별회계 순계다.
- 기금 포함 자료에는 순계가 없어 총계 대안으로만 사용한다.

In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

np.random.seed(42)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW_DIR = ROOT / "data/raw/지방재정365/기능별_회계별_세출예산"
FUND_RAW_DIR = ROOT / "data/raw/지방재정365/세출예산"
OUTPUT_PATH = ROOT / "data/processed/analysis/2016-2024_시도별_총세출_분모_패널.csv"
SENSITIVITY_PATH = ROOT / "data/processed/analysis/2016-2024_시도별_총세출_분모_민감도_패널.csv"
EXPECTED_ACCOUNTS = {"일반회계", "기타특별회계", "공기업특별회계"}
VALUE_COLUMNS = ["세출예산총계액", "세출예산순계액"]
FUND_ACCOUNT_COLUMNS = ["일반회계", "공기업특별회계", "기타특별회계", "기금"]

## Data

In [2]:
# 1. 원자료 로드
parts, source_manifest = [], []
for path in sorted(RAW_DIR.glob("*.csv")):
    frame = pd.read_csv(path, encoding="utf-8-sig")
    parts.append(frame)
    source_manifest.append(
        {
            "파일": path.name,
            "행": len(frame),
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        }
    )

raw = pd.concat(parts, ignore_index=True)
assert len(parts) == 9 and len(raw) == 44_897
assert set(raw["회계구분명"]) == EXPECTED_ACCOUNTS
display(pd.DataFrame(source_manifest)[["파일", "행"]])

fund_parts, fund_source_manifest = [], []
for path in sorted(FUND_RAW_DIR.glob("*.csv")):
    frame = pd.read_csv(path, encoding="utf-8-sig", dtype={"자치단체코드": "string"})
    fund_parts.append(frame)
    fund_source_manifest.append(
        {
            "파일": path.name,
            "행": len(frame),
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        }
    )

fund_raw = pd.concat(fund_parts, ignore_index=True)
assert len(fund_parts) == 9 and len(fund_raw) == 2_187
display(pd.DataFrame(fund_source_manifest)[["파일", "행"]])

,파일,행
0,2016_기능별 회계별 세출예산.csv,5077
1,2017_기능별 회계별 세출예산.csv,5037
2,2018_기능별 회계별 세출예산.csv,4914
3,2019_기능별 회계별 세출예산.csv,4958
4,2020_기능별 회계별 세출예산.csv,4962
5,2021_기능별 회계별 세출예산.csv,5011
6,2022_기능별 회계별 세출예산.csv,4978
7,2023_기능별 회계별 세출예산.csv,4968
8,2024_기능별 회계별 세출예산.csv,4992


,파일,행
0,2016_세출예산.csv,243
1,2017_세출예산.csv,243
2,2018_세출예산.csv,243
3,2019_세출예산.csv,243
4,2020_세출예산.csv,243
5,2021_세출예산.csv,243
6,2022_세출예산.csv,243
7,2023_세출예산.csv,243
8,2024_세출예산.csv,243


## Results

In [3]:
# 2. 지역·연도 집계
municipality_counts = (
    raw[["회계연도", "지역명", "자치단체코드"]]
    .drop_duplicates()
    .groupby(["회계연도", "지역명"])
    .size()
    .rename("자치단체수")
)
account_counts = raw.groupby(["회계연도", "지역명"])["회계구분명"].nunique().rename("포함회계수")

panel = (
    raw.groupby(["회계연도", "지역명"], as_index=False)[VALUE_COLUMNS]
    .sum()
    .merge(municipality_counts.reset_index(), on=["회계연도", "지역명"], validate="one_to_one")
    .merge(account_counts.reset_index(), on=["회계연도", "지역명"], validate="one_to_one")
    .rename(
        columns={
            "회계연도": "연도",
            "지역명": "지역",
            "세출예산총계액": "세출예산총계액_원",
            "세출예산순계액": "세출예산순계액_원",
        }
    )
    .sort_values(["연도", "지역"])
    .reset_index(drop=True)
)
panel["세출예산총계액_백만원"] = panel["세출예산총계액_원"] / 1_000_000
panel["세출예산순계액_백만원"] = panel["세출예산순계액_원"] / 1_000_000
panel["총계순계차액_백만원"] = panel["세출예산총계액_백만원"] - panel["세출예산순계액_백만원"]
panel["예산단계"] = "당초예산"
panel["포함회계"] = "일반회계+기타특별회계+공기업특별회계"
panel["원자료단위"] = "원"
panel["출처"] = "행정안전부 지방재정365 기능별 회계별 세출예산"

column_order = [
    "연도",
    "지역",
    "예산단계",
    "포함회계",
    "자치단체수",
    "포함회계수",
    "세출예산총계액_원",
    "세출예산순계액_원",
    "세출예산총계액_백만원",
    "세출예산순계액_백만원",
    "총계순계차액_백만원",
    "원자료단위",
    "출처",
]
panel = panel[column_order]
display(panel.head(17))

,연도,지역,예산단계,포함회계,자치단체수,포함회계수,세출예산총계액_원,세출예산순계액_원,세출예산총계액_백만원,세출예산순계액_백만원,총계순계차액_백만원,원자료단위,출처
0,2016,강원,당초예산,일반회계+기타특별회계+공기업특별회계,19,3,12439345023000,9415973114000,1.243935e+07,9.415973e+06,3.023372e+06,원,행정안전부 지방재정365 기능별 회계별 세출예산
1,2016,경기,당초예산,일반회계+기타특별회계+공기업특별회계,32,3,47124571518000,36249467525000,4.712457e+07,3.624947e+07,1.087510e+07,원,행정안전부 지방재정365 기능별 회계별 세출예산
2,2016,경남,당초예산,일반회계+기타특별회계+공기업특별회계,19,3,18897926368000,14026791169000,1.889793e+07,1.402679e+07,4.871135e+06,원,행정안전부 지방재정365 기능별 회계별 세출예산
3,2016,경북,당초예산,일반회계+기타특별회계+공기업특별회계,24,3,20737266442000,15225641095000,2.073727e+07,1.522564e+07,5.511625e+06,원,행정안전부 지방재정365 기능별 회계별 세출예산
4,2016,광주,당초예산,일반회계+기타특별회계+공기업특별회계,6,3,5917328537000,4106117093000,5.917329e+06,4.106117e+06,1.811211e+06,원,행정안전부 지방재정365 기능별 회계별 세출예산
5,2016,대구,당초예산,일반회계+기타특별회계+공기업특별회계,9,3,10142136000000,7213154474000,1.014214e+07,7.213154e+06,2.928982e+06,원,행정안전부 지방재정365 기능별 회계별 세출예산
6,2016,대전,당초예산,일반회계+기타특별회계+공기업특별회계,6,3,5636043167000,4016882713000,5.636043e+06,4.016883e+06,1.619160e+06,원,행정안전부 지방재정365 기능별 회계별 세출예산
7,2016,부산,당초예산,일반회계+기타특별회계+공기업특별회계,17,3,14696969244000,10573185713000,1.469697e+07,1.057319e+07,4.123784e+06,원,행정안전부 지방재정365 기능별 회계별 세출예산
8,2016,서울,당초예산,일반회계+기타특별회계+공기업특별회계,26,3,39210741290000,27534501217000,3.921074e+07,2.753450e+07,1.167624e+07,원,행정안전부 지방재정365 기능별 회계별 세출예산
9,2016,세종,당초예산,일반회계+기타특별회계+공기업특별회계,1,3,1117265816000,1048827518000,1.117266e+06,1.048828e+06,6.843830e+04,원,행정안전부 지방재정365 기능별 회계별 세출예산


In [4]:
# 3. 153행 계약 검사
expected_keys = pd.MultiIndex.from_product(
    [range(2016, 2025), sorted(panel["지역"].unique())], names=["연도", "지역"]
)
actual_keys = pd.MultiIndex.from_frame(panel[["연도", "지역"]])
qa = pd.DataFrame(
    [
        {
            "행": len(panel),
            "연도수": panel["연도"].nunique(),
            "지역수": panel["지역"].nunique(),
            "키누락": len(expected_keys.difference(actual_keys)),
            "키중복": int(panel.duplicated(["연도", "지역"]).sum()),
            "금액결측": int(panel.filter(like="예산").isna().any(axis=1).sum()),
            "0이하_순계액": int(panel["세출예산순계액_원"].le(0).sum()),
            "총계<순계": int(panel["총계순계차액_백만원"].lt(0).sum()),
        }
    ]
)
assert qa.iloc[0].to_dict() == {
    "행": 153,
    "연도수": 9,
    "지역수": 17,
    "키누락": 0,
    "키중복": 0,
    "금액결측": 0,
    "0이하_순계액": 0,
    "총계<순계": 0,
}
display(qa)

,행,연도수,지역수,키누락,키중복,금액결측,0이하_순계액,총계<순계
0,153,9,17,0,0,0,0,0


In [5]:
# 4. 저장 후 재검증
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
panel.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
reloaded = pd.read_csv(OUTPUT_PATH, encoding="utf-8-sig")
pd.testing.assert_frame_equal(reloaded, panel, check_dtype=False)
print("saved:", OUTPUT_PATH.relative_to(ROOT), "/", len(reloaded), "rows")

saved: data/processed/analysis/2016-2024_시도별_총세출_분모_패널.csv / 153 rows


In [6]:
# 5. 전국 연도별 순계액
national_totals = panel.groupby("연도", as_index=False)["세출예산순계액_원"].sum()
national_totals["전국순계액_조원"] = national_totals["세출예산순계액_원"] / 1_000_000_000_000
display(national_totals[["연도", "전국순계액_조원"]])

,연도,전국순계액_조원
0,2016,184.582527
1,2017,193.153245
2,2018,210.678389
3,2019,231.015190
4,2020,253.226288
5,2021,263.091736
6,2022,288.308320
7,2023,305.410895
8,2024,310.081825


### 기금 자료 교차검증

In [7]:
# 6. 세출예산 자료의 회계 합계와 기존 자료 역대조
fund_raw["3개회계총계_원"] = fund_raw[["일반회계", "공기업특별회계", "기타특별회계"]].sum(axis=1)
fund_raw["4개회계총계_원"] = fund_raw[FUND_ACCOUNT_COLUMNS].sum(axis=1)
raw["자치단체코드"] = raw["자치단체코드"].astype("string")

old_municipality_total = raw.groupby(["회계연도", "지역명", "자치단체코드"], as_index=False)[
    "세출예산총계액"
].sum()
crosscheck = old_municipality_total.merge(
    fund_raw[["회계연도", "지역명", "자치단체코드", "3개회계총계_원"]],
    on=["회계연도", "지역명", "자치단체코드"],
    how="outer",
    validate="one_to_one",
    indicator=True,
)
fund_qa = pd.DataFrame(
    [
        {
            "행": len(fund_raw),
            "키중복": int(fund_raw.duplicated(["회계연도", "자치단체코드"]).sum()),
            "결측": int(fund_raw.isna().sum().sum()),
            "음수": int(fund_raw[FUND_ACCOUNT_COLUMNS + ["세출총계"]].lt(0).sum().sum()),
            "4개회계합계불일치": int(fund_raw["4개회계총계_원"].ne(fund_raw["세출총계"]).sum()),
            "기존3개회계불일치": int(
                crosscheck["세출예산총계액"].ne(crosscheck["3개회계총계_원"]).sum()
            ),
            "조인누락": int(crosscheck["_merge"].ne("both").sum()),
        }
    ]
)
assert fund_qa.iloc[0].to_dict() == {
    "행": 2_187,
    "키중복": 0,
    "결측": 0,
    "음수": 0,
    "4개회계합계불일치": 0,
    "기존3개회계불일치": 0,
    "조인누락": 0,
}
display(fund_qa)

,행,키중복,결측,음수,4개회계합계불일치,기존3개회계불일치,조인누락
0,2187,0,0,0,0,0,0


In [8]:
# 7. 분모 대안 패널
def aggregate_variant(frame, value_column, variant, admin_scope, account_scope, value_basis):
    result = (
        frame.groupby(["회계연도", "지역명"], as_index=False)[value_column]
        .sum()
        .rename(columns={"회계연도": "연도", "지역명": "지역", value_column: "분모금액_원"})
    )
    result["분모대안"] = variant
    result["행정범위"] = admin_scope
    result["회계범위"] = account_scope
    result["금액기준"] = value_basis
    return result


headquarters = raw.loc[raw["자치단체명"].astype("string").str.endswith("본청")].copy()
headquarter_keys = headquarters[["회계연도", "지역명", "자치단체코드"]].drop_duplicates()
assert headquarter_keys.groupby(["회계연도", "지역명"]).size().eq(1).all()

variants = [
    aggregate_variant(
        raw,
        "세출예산순계액",
        "지역통합_3개회계_순계",
        "광역본청+기초",
        "일반+기타특별+공기업특별",
        "순계",
    ),
    aggregate_variant(
        raw.loc[raw["회계구분명"].eq("일반회계")],
        "세출예산순계액",
        "지역통합_일반회계_순계",
        "광역본청+기초",
        "일반회계",
        "순계",
    ),
    aggregate_variant(
        raw,
        "세출예산총계액",
        "지역통합_3개회계_총계",
        "광역본청+기초",
        "일반+기타특별+공기업특별",
        "총계",
    ),
    aggregate_variant(
        headquarters,
        "세출예산순계액",
        "광역본청_3개회계_순계",
        "광역본청",
        "일반+기타특별+공기업특별",
        "순계",
    ),
    aggregate_variant(
        fund_raw,
        "세출총계",
        "지역통합_4개회계_총계",
        "광역본청+기초",
        "일반+기타특별+공기업특별+기금",
        "총계",
    ),
]
sensitivity = pd.concat(variants, ignore_index=True)
sensitivity["분모금액_백만원"] = sensitivity["분모금액_원"] / 1_000_000
sensitivity["예산단계"] = "당초예산"
sensitivity["출처"] = np.where(
    sensitivity["분모대안"].eq("지역통합_4개회계_총계"),
    "행정안전부 지방재정365 세출예산",
    "행정안전부 지방재정365 기능별 회계별 세출예산",
)
sensitivity = (
    sensitivity[
        [
            "연도",
            "지역",
            "분모대안",
            "예산단계",
            "행정범위",
            "회계범위",
            "금액기준",
            "분모금액_원",
            "분모금액_백만원",
            "출처",
        ]
    ]
    .sort_values(["분모대안", "연도", "지역"])
    .reset_index(drop=True)
)
display(
    sensitivity.groupby("분모대안").agg(
        행=("분모금액_원", "size"), 최솟값_백만원=("분모금액_백만원", "min")
    )
)

,행,최솟값_백만원
분모대안,,
광역본청_3개회계_순계,153,1048827.518
지역통합_3개회계_순계,153,1048827.518
지역통합_3개회계_총계,153,1117265.816
지역통합_4개회계_총계,153,1194522.501
지역통합_일반회계_순계,153,753684.202


In [9]:
# 8. 대안별 153키 QA와 저장
variant_qa = (
    sensitivity.groupby("분모대안")
    .apply(
        lambda group: pd.Series(
            {
                "행": len(group),
                "연도수": group["연도"].nunique(),
                "지역수": group["지역"].nunique(),
                "키중복": int(group.duplicated(["연도", "지역"]).sum()),
                "금액결측": int(group["분모금액_원"].isna().sum()),
                "0이하": int(group["분모금액_원"].le(0).sum()),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
assert variant_qa[["행", "연도수", "지역수"]].eq([153, 9, 17]).all().all()
assert variant_qa[["키중복", "금액결측", "0이하"]].to_numpy().sum() == 0

SENSITIVITY_PATH.parent.mkdir(parents=True, exist_ok=True)
sensitivity.to_csv(SENSITIVITY_PATH, index=False, encoding="utf-8-sig")
reloaded_sensitivity = pd.read_csv(SENSITIVITY_PATH, encoding="utf-8-sig")
pd.testing.assert_frame_equal(reloaded_sensitivity, sensitivity, check_dtype=False)
display(variant_qa)
print("saved:", SENSITIVITY_PATH.relative_to(ROOT), "/", len(sensitivity), "rows")

,분모대안,행,연도수,지역수,키중복,금액결측,0이하
0,광역본청_3개회계_순계,153,9,17,0,0,0
1,지역통합_3개회계_순계,153,9,17,0,0,0
2,지역통합_3개회계_총계,153,9,17,0,0,0
3,지역통합_4개회계_총계,153,9,17,0,0,0
4,지역통합_일반회계_순계,153,9,17,0,0,0


saved: data/processed/analysis/2016-2024_시도별_총세출_분모_민감도_패널.csv / 765 rows


## Takeaways

- 주 기준과 4개 대안, 총 765행을 저장했다.
- 기금 포함 대안은 자료 제약상 총계 기준이다.
- 두 원자료의 3개 회계 총계는 2,187키에서 모두 일치한다.